# Multiclass v2 · lima pretrained encoder
**Eksperimen normalisasi per model + enam teknik post-processing.**

Kaggle GPU, internet aktif. Lima model yang diminta, masing-masing dua input: **native** vs **normalisasi slang+elongation terkurasi**. Total 10 fine-tuning; budget epoch/panjang token sama, tokenizer sesuai model. TF-IDF/Word2Vec tidak digunakan pada v2.

| Model | Model ID | Input native |
|---|---|---|
| IndoBERTweet Base | indolem/indobertweet-base-uncased | Lowercase, @USER/HTTPURL, emoji ke teks |
| IndoBERT Base | indobenchmark/indobert-base-p1 | Safe cleaning, lowercase |
| XLM-R Base | FacebookAI/xlm-roberta-base | Safe cleaning, pertahankan case/emoji/bahasa |
| IndoBERT IndoLEM | indolem/indobert-base-uncased | Safe cleaning, lowercase |
| NusaBERT Base | LazarusNLP/NusaBERT-base | Safe cleaning, lowercase |

**Normalisasi** hanya mengganti slang yang sudah dikurasi dan elongation yang cocok whitelist. Setelahnya tetap mengikuti recipe model masing-masing. Tidak menerjemahkan bahasa daerah, membuang negasi/angka, stemming, atau stopword removal. Manfaat diuji; varian normalized tidak otomatis dipilih.

**Data/split:** benchmark MBG yang sama dengan v1, dengan split v2 deterministik yang menyatukan pengguna, duplikat safe, dan collision hasil normalisasi sebelum split. Dataset ini sudah dieksplorasi di v1; holdout benchmark ini bukan test eksternal baru atau perbandingan berpasangan persis dengan v1. Semua keputusan v2 memakai validation; tabel holdout dibuka setelah keputusan dibekukan. Label contoh diperlakukan sebagai pseudo-label, bukan anotasi sentimen manusia.

**Post-processing:** threshold gate multiclass, bias kelas, probability ensemble, konsistensi duplikat, rule terverifikasi train, temperature calibration; tambahan ensemble+bias. Cross-fit validation mengurangi overfit parameter, tetapi tetap skor pengembangan karena pemilihan model/normalisasi juga memakai validation.

## 00 · Konfigurasi

Sekitar 5.000 komentar; ubah DATA_FILE/kolom untuk data lain. Satu seed dan pencarian terbatas tidak membuktikan optimum global.

In [1]:
from pathlib import Path
SEED,N_ROWS,EDA_N,TOP=42,5000,5000,15
DATA_FILE=None
TEXT_COL,LABEL_COL,CONF_COL="Comment","Sentiment","Confidence Score"
SOURCE_COL,GROUP_COL=None,"Commenter ID"
NEAR_DUP,NEAR_THRESHOLD=True,.9
EPOCHS,MAX_LEN,LEARNING_RATE=3,192,2e-5
WEIGHT_POWER=.5  # akar inverse-frequency dari train; lebih lunak daripada balanced penuh
POST_FOLDS,MIN_POST_GAIN,ENSEMBLE_CANDIDATES=3,.003,64
RULE_MIN_SUPPORT,RULE_MIN_PRECISION=8,.95
OUT=Path("/kaggle/working/hasil_v2")

In [2]:
MODEL_SPECS=[
 {"name":"01_indobertweet","id":"indolem/indobertweet-base-uncased","family":"tweet","batch":32,"accum":1,"revision":"32e28c05b47e33b6675d2670a1745c50a65e987a"},
 {"name":"02_indobert_p1","id":"indobenchmark/indobert-base-p1","family":"uncased","batch":32,"accum":1,"revision":"c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2"},
 {"name":"03_xlm_roberta","id":"FacebookAI/xlm-roberta-base","family":"cased","batch":16,"accum":2,"revision":"e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"},
 {"name":"04_indobert_indolem","id":"indolem/indobert-base-uncased","family":"uncased","batch":32,"accum":1,"revision":"7ccb3cd0f5b08ffbaa465aade22328e8600e23eb"},
 {"name":"05_nusabert","id":"LazarusNLP/NusaBERT-base","family":"uncased","batch":32,"accum":1,"revision":"e52beecf63a65daeae103c4fe8ad6edbf3e9743b"},
]

## 01 · Lingkungan Kaggle

Dependensi dipasang di Kaggle. Model public diambil pada revision yang dipin; tidak ada token akses dalam notebook.

In [3]:
import sys,os,subprocess
assert Path("/kaggle/input").exists(),"Jalankan di Kaggle GPU."
os.environ.update(USE_TF="0",USE_FLAX="0",TOKENIZERS_PARALLELISM="false",HF_HUB_DISABLE_PROGRESS_BARS="1",
    HF_HUB_ETAG_TIMEOUT="60",HF_HUB_DOWNLOAD_TIMEOUT="300")
subprocess.run([sys.executable,"-m","pip","install","-q","ftfy>=6.3,<7","emoji>=2.14,<3",
    "datasketch>=1.6,<3","langid==1.1.6","sentencepiece>=0.2,<1"],check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'ftfy>=6.3,<7', 'emoji>=2.14,<3', 'datasketch>=1.6,<3', 'langid==1.1.6', 'sentencepiece>=0.2,<1'], returncode=0)

In [4]:
import json,time,random,re,html,unicodedata,gc,hashlib
from collections import Counter
import numpy as np
import pandas as pd
import torch,emoji,ftfy
from IPython.display import display
from sklearn.metrics import f1_score,accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
assert torch.cuda.is_available(), "Aktifkan GPU Kaggle untuk menjalankan kelima pendekatan."
OUT.mkdir(exist_ok=True)
print("GPU:",torch.cuda.get_device_name(0))

GPU: Tesla T4


In [5]:
def show(title,table):
    print("\n"+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(3))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({"n":n,"%":n/(len(values) or np.nan)*100})

def macro(y,p):return f1_score(y,p,labels=np.arange(C),average="macro",zero_division=0)
pd.set_option("display.max_columns",None,"display.max_colwidth",130)
RUN_START=time.perf_counter()

## 02 · Data mentah & kualitas

Sebelum drop missing/invalid, audit seluruh CSV. Selanjutnya N_ROWS membatasi benchmark. Tidak ada data demo sintetis.

In [6]:
INPUT=Path("/kaggle/input")
path=Path(DATA_FILE) if DATA_FILE else next(INPUT.rglob("comment_sentiment.csv"))
raw=pd.read_csv(path)
assert {TEXT_COL,LABEL_COL}.issubset(raw), f"Kolom tersedia: {raw.columns.tolist()}"
show("Skema sebelum pembersihan",pd.DataFrame({"tipe":raw.dtypes.astype(str),"missing":raw.isna().sum(),
    "missing_%":raw.isna().mean()*100,"unik_non_null":raw.nunique(dropna=True)}))
print("File:",path,"| total baris:",len(raw),"| baris duplikat ekstra:",raw.duplicated().sum())


Skema sebelum pembersihan


,tipe,missing,missing_%,unik_non_null
Comment ID,object,0,0.0,5034
Comment Link,object,0,0.0,5034
Commenter Name,object,0,0.0,4431
Commenter ID,object,0,0.0,4431
Comment,object,0,0.0,4996
Likes,int64,0,0.0,45
Cleaned Comment,object,0,0.0,4996
Sentiment,object,0,0.0,3
Confidence Score,float64,0,0.0,4966


File: /kaggle/input/datasets/dzikriraihan/indonesian-youtube-comments-mbg-program/comment_sentiment.csv | total baris: 5035 | baris duplikat ekstra: 1


In [7]:
raw_text=raw[TEXT_COL].where(raw[TEXT_COL].map(lambda x:isinstance(x,str)),"").fillna("")
quality=pd.DataFrame({"teks_null":raw[TEXT_COL].isna(),"teks_non_string":raw[TEXT_COL].notna() & ~raw[TEXT_COL].map(lambda x:isinstance(x,str)),
    "teks_kosong":raw_text.str.strip().eq(""),"label_null":raw[LABEL_COL].isna(),
    "label_kosong":raw[LABEL_COL].astype("string").str.strip().eq("").fillna(False),
    "teks_pseudo_null":raw_text.str.strip().str.casefold().isin(["nan","null","none","n/a","-"])})
show("Kualitas input · flag dapat overlap",pd.DataFrame({"n":quality.sum(),"%":quality.mean()*100}))
valid=~quality.any(axis=1)
data=raw.loc[valid].copy().reset_index(names="row_id")
data[LABEL_COL]=data[LABEL_COL].astype(str).str.strip()
if N_ROWS and len(data)>N_ROWS:data=data.sample(N_ROWS,random_state=SEED).reset_index(drop=True)
print("Dikeluarkan karena invalid:",int((~valid).sum()),"| baris pilot:",len(data))


Kualitas input · flag dapat overlap


,n,%
teks_null,0,0.0
teks_non_string,0,0.0
teks_kosong,0,0.0
label_null,0,0.0
label_kosong,0,0.0
teks_pseudo_null,0,0.0


Dikeluarkan karena invalid: 0 | baris pilot: 5000


## 03 · Preprocessing universal · bagian yang sesuai

Fungsi berikut diambil langsung dari stepwise.py. Native dan normalized berbeda hanya pada dua langkah normalisasi kata; recipe model tidak berubah antara dua varian.

In [8]:
SLANG = dict(zip(
    'gk ga gak nggak ngga tdk blm jgn yg dgn utk krn karna bgt tp udh sdh skrg gmn klo kalo jg bgs'.split(),
    'tidak tidak tidak tidak tidak tidak belum jangan yang dengan untuk karena karena banget tapi sudah sudah sekarang bagaimana kalau kalau juga bagus'.split()))
PROTECTED = set('mbg sppg bgn apbn apbd bpom bpk umkm allah massa jakarta bandung'.split())
NEGATIONS = set('tidak tak bukan belum jangan tanpa kurang nggak gak ga gk ora teu indak'.split())
AMBIGUOUS = set('sm dr bs pd lg org'.split())
VOCAB = set('bagus banget lama makanan makan bergizi gratis program keracunan anggaran korupsi sehat sekolah anak siswa pemerintah merata enak'.split()) | set(SLANG.values()) | PROTECTED
TOKEN = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+|[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}|(?<!\w)[@#][\w.]+|[^\W\d_]+', re.I)
WORD = re.compile(r'[^\W\d_]+')
URL = re.compile(r'https?://[^\s<>]+|www\.[^\s<>]+', re.I)

In [9]:
def map_words(text, transform):
    return TOKEN.sub(lambda m: transform(m[0]) if WORD.fullmatch(m[0]) else m[0], text)

def valid_text(value):
    # None/NaN/pd.NA/numbers/list are not converted to literal "nan"/"123".
    return value if isinstance(value, str) else ''

def repair_encoding(text):
    return ftfy.fix_text(text, unescape_html=False, uncurl_quotes=False,
                        fix_latin_ligatures=False, fix_character_width=False,
                        remove_control_chars=False, normalization=None)

def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)  # retain emoji/local scripts/fullwidth

In [10]:
def decode_html_entities(text):
    return html.unescape(text)

def remove_invisible(text):
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    return re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [11]:
def normalize_slang(text, mapping=SLANG):
    return map_words(text, lambda w: mapping.get(w.casefold(), w)
                     if w.casefold() not in PROTECTED | AMBIGUOUS else w)

def normalize_elongation(text, vocab=VOCAB):
    def shorten(word):
        if word.casefold() in PROTECTED:
            return word
        candidates = [re.sub(r'([^\W\d_])\1{2,}', replacement, word, flags=re.I)
                      for replacement in (r'\1', r'\1\1')]
        return next((w for w in candidates if w.casefold() in vocab), word)
    return map_words(text, shorten)

def indobertweet_view(text):
    text = URL.sub('HTTPURL', text.lower())
    text = re.sub(r'(?<![\w@])@[\w.]+', '@USER', text)
    return normalize_whitespace(emoji.demojize(text))

In [12]:
def safe_clean(value):
    for function in [valid_text,repair_encoding,normalize_unicode,decode_html_entities,remove_invisible,normalize_whitespace]:value=function(value)
    return value
cache={s:safe_clean(s) for s in data[TEXT_COL].unique()}
data["safe"]=data[TEXT_COL].map(cache)
print("Kosong setelah safe:",data.safe.eq("").sum())
data=data.loc[data.safe.ne("")].reset_index(drop=True)
cache={s:normalize_elongation(normalize_slang(s)) for s in data.safe.unique()}
data["normal"]=data.safe.map(cache)

def model_input(frame,spec,view):
    texts=frame["normal" if view=="normalized" else "safe"]
    if spec["family"]=="tweet":return texts.map(indobertweet_view)
    return texts.str.lower() if spec["family"]=="uncased" else texts

Kosong setelah safe: 0


## 04 · Split grup yang sama untuk semua eksperimen

Pengguna yang sama atau teks duplikat/near-duplicate tidak boleh melintasi split. Kandidat near-duplicate diverifikasi Jaccard; deteksi ini tetap aproksimasi.

In [13]:
def duplicate_groups(frame):
    from datasketch import MinHash,MinHashLSH
    keys=frame.safe.str.casefold().str.replace(r"\s+"," ",regex=True)
    parent=list(range(len(frame)))
    def root(i):
        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
        return i
    def merge(i,j):parent[root(i)]=root(j)
    first={}
    for i,key in enumerate(keys):
        if key in first:merge(i,first[key])
        else:first[key]=i
    normal_first={}
    for i,key in enumerate(frame.normal.str.casefold().str.replace(r"\s+"," ",regex=True)):
        if key in normal_first:merge(i,normal_first[key])
        else:normal_first[key]=i
    if GROUP_COL:
        assert GROUP_COL in frame, f"Kolom grup tidak ada: {GROUP_COL}"
        for ids in frame.groupby(GROUP_COL).indices.values():
            for i in ids[1:]:merge(int(ids[0]),int(i))
    if NEAR_DUP:
        index=MinHashLSH(threshold=NEAR_THRESHOLD,num_perm=64)
        shingles={}
        for key,i in first.items():
            if len(key)<20:continue
            grams={key[j:j+5] for j in range(len(key)-4)}
            signature=MinHash(num_perm=64,seed=SEED)
            signature.update_batch([g.encode() for g in sorted(grams)])
            for candidate in index.query(signature):
                other=shingles[candidate]
                if len(grams&other)/len(grams|other)>=NEAR_THRESHOLD:merge(i,int(candidate))
            index.insert(str(i),signature);shingles[str(i)]=grams
    components=np.array([root(i) for i in range(len(frame))])
    return pd.Series(np.arange(len(frame))).groupby(components).transform("min").to_numpy()

In [14]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder().fit(data[LABEL_COL]);classes=encoder.classes_;C=len(classes)
assert C>=3, "Pilot ini untuk klasifikasi multiclass."
data["y"]=encoder.transform(data[LABEL_COL])
data["dup_group"]=duplicate_groups(data)
assert data.groupby("y").dup_group.nunique().min()>=5, "Perlu >=5 grup independen per kelas; ubah skema split untuk data kecil."
fold=np.full(len(data),-1)
for k,(_,idx) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(data,data.y,data.dup_group)):fold[idx]=k
train=data.loc[fold>=2].copy();val=data.loc[fold==1].copy();holdout=data.loc[fold==0].copy()
for a,b in [(train,val),(train,holdout),(val,holdout)]:assert set(a.dup_group).isdisjoint(b.dup_group)
for frame in [train,val,holdout]:assert set(frame.y)==set(range(C)), "Ada kelas hilang pada split; revisi skema sebelum training."
show("Pembagian baris",pd.DataFrame({name:frame[LABEL_COL].value_counts() for name,frame in [("train",train),("validation",val),("holdout",holdout)]}).fillna(0))
print("Grup independen:",data.dup_group.nunique(),"| overlap grup antar-split: 0")


Pembagian baris


,train,validation,holdout
Sentiment,,,
negative,2089,676,690
positive,683,221,220
neutral,247,101,73


Grup independen: 4381 | overlap grup antar-split: 0


In [15]:
split=pd.concat([frame[["row_id","dup_group"]].assign(split=name) for name,frame in [("train",train),("validation",val),("holdout",holdout)]])
split.to_csv(OUT/"split.csv",index=False)
print("SHA split:",hashlib.sha256(split.sort_values("row_id").to_csv(index=False).encode()).hexdigest())

SHA split: 6bd611a55f0907b9917ba0e4381c491044210762a926cd62ddba75a29ec34456


## 05 · EDA train · angka saja

Missing/skema ada sebelum drop pada 02. Di sini: class distribution, duplikat, length/kuantil, hapax, top words/ngram, noise per kelas, konflik label, confidence pemberi label, contoh, dan estimasi bahasa. Bahasa estimasi tidak diperlakukan sebagai ground truth.

In [16]:
t=train[TEXT_COL].astype(str);words=t.str.split().str.len();chars=t.str.len()
key=train.safe.str.casefold().str.replace(r"\s+"," ",regex=True)
show("Distribusi kelas train",distribution(train[LABEL_COL]))
show("Panjang karakter/kata",pd.DataFrame({"karakter":chars,"kata":words}).describe(percentiles=[.01,.05,.5,.9,.95,.99,.999]).T)
show("Duplikat train",pd.Series({"persis_ekstra":t.duplicated().sum(),"normal_ekstra":key.duplicated().sum(),
    "normal_ekstra_%":key.duplicated().mean()*100,"grup_dup":key.value_counts().gt(1).sum()}))
show("Panjang per kelas",words.groupby(train[LABEL_COL]).describe(percentiles=[.5,.95,.99]))


Distribusi kelas train


,n,%
Sentiment,,
negative,2089,69.195
positive,683,22.623
neutral,247,8.182



Panjang karakter/kata


,count,mean,std,min,1%,5%,50%,90%,95%,99%,99.9%,max
karakter,3019.0,191.647,325.195,1.0,6.0,17.0,110.0,389.2,580.2,1357.20,4256.050,7210.0
kata,3019.0,28.550,46.500,1.0,1.0,3.0,17.0,58.0,85.0,199.74,565.442,1119.0



Duplikat train


,0
persis_ekstra,15.000
normal_ekstra,19.000
normal_ekstra_%,0.629
grup_dup,14.000



Panjang per kelas


,count,mean,std,min,50%,95%,99%,max
Sentiment,,,,,,,,
negative,2089.0,32.251,53.283,1.0,19.0,101.0,238.60,1119.0
neutral,247.0,10.692,9.711,1.0,8.0,29.7,45.32,74.0
positive,683.0,23.690,25.710,1.0,16.0,66.0,124.90,250.0


In [17]:
sample=t.sample(min(EDA_N,len(t)),random_state=SEED)
seqs=sample.str.casefold().str.findall(r"[^\W\d_]+")
counts=pd.Series(Counter(w for seq in seqs for w in seq),dtype="int64").sort_values(ascending=False)
docs=pd.Series(Counter(w for seq in seqs for w in set(seq)),dtype="int64")
show("Vocabulary train · sampel",pd.Series({"dokumen":len(seqs),"token":counts.sum(),"unik":len(counts),
    "TTR_%":100*len(counts)/(counts.sum() or np.nan),"hapax":counts.eq(1).sum(),"hapax_%_tipe":100*counts.eq(1).mean(),
    "hapax_%_token":100*counts.eq(1).sum()/(counts.sum() or np.nan),"langka_<=5_%_tipe":100*counts.le(5).mean()}))
show("Top words",counts.head(TOP).rename("n").to_frame().assign(pct_token=counts.head(TOP)/(counts.sum() or np.nan)*100,
    dokumen=docs.reindex(counts.head(TOP).index),pct_dokumen=docs.reindex(counts.head(TOP).index)/len(seqs)*100))


Vocabulary train · sampel


,0
dokumen,3019.000
token,89656.000
unik,10495.000
TTR_%,11.706
hapax,5763.000
hapax_%_tipe,54.912
hapax_%_token,6.428
langka_<=5_%_tipe,82.039



Top words


,n,pct_token,dokumen,pct_dokumen
mbg,2143,2.390,1378,45.644
di,1781,1.986,967,32.030
br,1702,1.898,498,16.496
dan,1658,1.849,892,29.546
yg,1583,1.766,862,28.553
yang,1304,1.454,586,19.410
itu,850,0.948,563,18.649
ini,838,0.935,550,18.218
ada,726,0.810,509,16.860
tidak,670,0.747,464,15.369


In [18]:
for n in [2,3]:
    bag=pd.Series(Counter(" ".join(seq[i:i+n]) for seq in seqs for i in range(len(seq)-n+1)),dtype="int64").sort_values(ascending=False)
    show(f"Top {n}-gram tanpa melompati stopword",bag.head(TOP).rename("n").to_frame().assign(pct=bag.head(TOP)/(bag.sum() or np.nan)*100))
show("Contoh hapax",counts[counts.eq(1)].head(TOP))
odd=counts.index.to_series().str.len().ge(20)|counts.index.to_series().str.count(r"([^\W\d_])\1{2,}").gt(0)
show("Kandidat kata panjang/berulang · bukan pasti typo",counts[odd].head(TOP))


Top 2-gram tanpa melompati stopword


,n,pct
br br,365,0.421
mbg itu,132,0.152
prof gema,124,0.143
program mbg,112,0.129
mbg di,92,0.106
pak gema,85,0.098
mbg ini,85,0.098
lapangan kerja,82,0.095
a href,75,0.087
lebih baik,74,0.085



Top 3-gram tanpa melompati stopword


,n,pct
www youtube com,64,0.076
https www youtube,50,0.060
youtube com watch,50,0.060
com watch v,50,0.060
a href https,48,0.057
href https www,48,0.057
amp t a,46,0.055
j amp t,46,0.055
xhep j amp,46,0.055
watch v svi,46,0.055



Contoh hapax


,0
kamuflase,1
ngeloloh,1
kehujanan,1
belaka,1
guling,1
jelasssss,1
jelasss,1
bembuka,1
laenya,1
banyank,1



Kandidat kata panjang/berulang · bukan pasti typo


,0
www,64
yaaa,6
pppk,4
astronacciinternational,4
dipertanggungjawabkan,4
hmmm,3
rakyatkayagakakanmenganggapindonesiabangkrutkarenamerekapunyauang,2
jossss,2
indonesiamenujubangkrutuntukrakyatmiskin,2
preeeeeeeettttttt,2


In [19]:
NOISE={"url":r"https?://|www\.","mention":r"@\w+","hashtag":r"#\w+","angka":r"\d",
    "emoji/simbol":r"[\U0001F300-\U0001FAFF\u2600-\u27BF]","kapital":r"\b[A-Z]{3,}\b",
    "huruf_ulang":r"([^\W\d_])\1{2,}","baca_ulang":r"[!?.,]{2,}","html":r"</?[a-zA-Z][^>]*>",
    "encoding":r"\ufffd|Ã.|Â.","invisible":r"[\u200b\ufeff]","newline":r"[\n\r]",
    "email":r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}","non_ASCII":r"[^\x00-\x7f]",
    "html_entity":r"&(?:#\d+|[a-zA-Z]+);","spasi_tepi":r"^\s|\s$","spasi_ulang":r" {2,}",
    "kontrol":r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]","huruf_angka":r"\b(?=\w*[a-zA-Z])(?=\w*\d)\w+\b",
    "slang_contoh":r"\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b",
    "negasi":r"\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|gk)\b"}
flags=pd.DataFrame({name:t.str.count(pattern,flags=0 if name=="kapital" else re.I).gt(0) for name,pattern in NOISE.items()})
show("Noise train · % seluruh baris, flag overlap",pd.DataFrame({"n":flags.sum(),"%":flags.mean()*100}).sort_values("n",ascending=False))
show("Noise per kelas (%)",flags.groupby(train[LABEL_COL]).mean()*100)
if SOURCE_COL:
    assert SOURCE_COL in train
    show("Sumber",distribution(train[SOURCE_COL]));show("Label per sumber (%)",pd.crosstab(train[SOURCE_COL],train[LABEL_COL],normalize="index")*100)


Noise train · % seluruh baris, flag overlap


,n,%
kapital,1305,43.226
slang_contoh,1166,38.622
negasi,1116,36.966
angka,930,30.805
baca_ulang,867,28.718
huruf_angka,622,20.603
non_ASCII,551,18.251
html,548,18.152
emoji/simbol,484,16.032
spasi_ulang,466,15.436



Noise per kelas (%)


,url,mention,hashtag,angka,emoji/simbol,kapital,huruf_ulang,baca_ulang,html,encoding,invisible,newline,email,non_ASCII,html_entity,spasi_tepi,spasi_ulang,kontrol,huruf_angka,slang_contoh,negasi
Sentiment,,,,,,,,,,,,,,,,,,,,,
negative,1.628,0.144,1.101,33.270,13.499,46.912,6.702,31.307,18.813,0.0,0.000,0.144,0.0,15.749,8.281,0.0,16.659,0.0,22.882,43.274,43.801
neutral,5.263,0.405,1.619,22.672,12.146,37.247,8.097,10.526,12.955,0.0,0.000,0.000,0.0,12.955,6.883,0.0,11.336,0.0,13.765,14.980,9.717
positive,1.025,0.439,0.439,26.208,25.183,34.114,4.685,27.379,18.009,0.0,0.146,0.000,0.0,27.818,6.149,0.0,13.177,0.0,16.105,32.943,25.915


In [20]:
conflict=train.groupby(key)[LABEL_COL].nunique().loc[lambda s:s.gt(1)]
print("Grup teks dengan konflik label di train:",len(conflict))
show("Contoh konflik · perlu review konteks",train.loc[key.isin(conflict.index),[TEXT_COL,LABEL_COL]].head(TOP))
if CONF_COL in train:
    confidence=pd.to_numeric(train[CONF_COL],errors="coerce")
    show("Confidence pemberi label per kelas",confidence.groupby(train[LABEL_COL]).describe(percentiles=[.1,.5,.9]))
show("Contoh teks per kelas · train",train.groupby(LABEL_COL)[TEXT_COL].apply(lambda s:s.sample(min(3,len(s)),random_state=SEED).str.slice(0,180)).to_frame("teks"))

Grup teks dengan konflik label di train: 1

Contoh konflik · perlu review konteks


,Comment,Sentiment
474,setuju,negative
3340,Setuju,positive



Confidence pemberi label per kelas


,count,mean,std,min,10%,50%,90%,max
Sentiment,,,,,,,,
negative,2089.0,0.919,0.134,0.401,0.686,0.988,0.999,1.000
neutral,247.0,0.782,0.179,0.382,0.506,0.816,0.987,0.999
positive,683.0,0.832,0.171,0.345,0.540,0.908,0.991,0.999



Contoh teks per kelas · train


teks
Sentiment                                                                                                                                        
negative  1267                                                                                                  Nye nye nye nye,, nye nye nye nye
          3547         bukan programnya...yg dadi masalah cara kinerjanja ...penempatan dapurnya....penyalah gunaan dana ..terutama yg di korupsi
          901   program omom omon minim kajian ya begitulah sifat pemerintah kita dari jaman pria solo, modelnya sama kaya IKN udah di bangun ...
neutral   768                                                                                                                    cakep pak gemma❤
          183                                                                                                                      Proff gema sip
          3628                                                                                                       Sehat- sehat terus prof gema
positive  2112                                                Program bagus dr presiden,cuma para pelaksana aja yg kurang amanah di tingkat bawah
          1035                                                               in this economy, di kota mana yg ada harga sekali makan 13rb - 15rb?
          1520  MBG sebenarnya bagus, cuma emang bener yang salah itu oknum, petani dan pedagang di tempat ku juga sangat terbantu. Aku yang d...

In [21]:
from langid.langid import LanguageIdentifier,model as language_model
from threadpoolctl import threadpool_limits
language_sample=sample.sample(min(300,len(sample)),random_state=SEED)
identifier=LanguageIdentifier.from_modelstring(language_model,norm_probs=True)
with threadpool_limits(limits=1):
    language=language_sample.map(lambda s:identifier.classify(s[:3000]) if len(s.split())>=5 else ("tidak_pasti",np.nan))
lang=language.map(lambda p:p[0] if p[1]>=.8 else "tidak_pasti")
show("Estimasi bahasa · % sampel, satu bahasa dominan per dokumen",distribution(lang))
print("Bahasa serumpun/campur kode/teks pendek dapat salah; tidak_pasti bukan bahasa Indonesia otomatis.")


Estimasi bahasa · % sampel, satu bahasa dominan per dokumen


,n,%
Comment,,
id,161,53.667
tidak_pasti,102,34.000
ms,25,8.333
es,3,1.000
jv,2,0.667
tl,1,0.333
km,1,0.333
sw,1,0.333
am,1,0.333


Bahasa serumpun/campur kode/teks pendek dapat salah; tidak_pasti bukan bahasa Indonesia otomatis.


In [22]:
show("Audit normalisasi train",pd.DataFrame({"berubah_%":[train.normal.ne(train.safe).mean()*100],
    "kosong":[train.normal.eq("").sum()],"unik_safe":[train.safe.nunique()],"unik_normal":[train.normal.nunique()]}))
show("Contoh perubahan",train.loc[train.normal.ne(train.safe),["safe","normal"]].head(TOP))
REGIONAL_MARKERS=set("ora mboten kowe piye durung ngono saiki teu henteu abdi kumaha naon ngeunah indak rancak lamak bana".split())
show("Penanda regional · indikasi saja",pd.Series({"n_train":train.safe.str.casefold().str.findall(r"[^\W\d_]+").map(lambda s:len(set(s)&REGIONAL_MARKERS)>=2).sum()}))


Audit normalisasi train


,berubah_%,kosong,unik_safe,unik_normal
0,45.048,0,3004,3004



Contoh perubahan


,safe,normal
1,Analisis paling guooblook krn mengabaikan norma dan tata nilai,Analisis paling guooblook karena mengabaikan norma dan tata nilai
2,Mbg adalah tdk merugikan siswa dan masarakat yg merugikan mbg adalah pengelolanya bukan presidenya,Mbg adalah tidak merugikan siswa dan masarakat yang merugikan mbg adalah pengelolanya bukan presidenya
7,"Master Gema saya sll menonton hampir semua podcast Master Gema baik ttf pasar modal ataupun ekonomi NKRI, saya setuju kita men...","Master Gema saya sll menonton hampir semua podcast Master Gema baik ttf pasar modal ataupun ekonomi NKRI, saya setuju kita men..."
11,Nyata nya hanya 8 rebu yg di sediakan,Nyata nya hanya 8 rebu yang di sediakan
12,"Masalah nya dana MBG diambil dari dana PENDIDIKAN. Dan knp kok korupsi di MBG lama sekali di tangani, kan sdh banyak yg info k...","Masalah nya dana MBG diambil dari dana PENDIDIKAN. Dan knp kok korupsi di MBG lama sekali di tangani, kan sudah banyak yang in..."
16,"Emang tujuan mereka ingin menghentikan MBG untuk menghancurkan Indonesia.<br>Nanti mereka akan teriak lagi, PRABOWO sudah gaga...","Emang tujuan mereka ingin menghentikan MBG untuk menghancurkan Indonesia.<br>Nanti mereka akan teriak lagi, PRABOWO sudah gaga..."
18,Barang busuk dibungkus tetap akan buruk efeknya kalau ketahuan atau terbuka percayalah bapak Prabowo yg terhormat,Barang busuk dibungkus tetap akan buruk efeknya kalau ketahuan atau terbuka percayalah bapak Prabowo yang terhormat
30,"Saya pedagang pasar tradisional, penghasilan bisa di katakan lumayan lah, saya punya anak umur 3 thn, jujur anak saya stiap ka...","Saya pedagang pasar tradisional, penghasilan bisa di katakan lumayan lah, saya punya anak umur 3 thn, jujur anak saya stiap ka..."
32,"Yg merugikan negara bukan mbg nya tetapi para pejabat yg korupsi itu, sekira para pejabat di mbg tdk rakus, tentu berguna bagi...","yang merugikan negara bukan mbg nya tetapi para pejabat yang korupsi itu, sekira para pejabat di mbg tidak rakus, tentu bergun..."
33,"Penyalurannya jg hrs lebih selektif , benar"" yg membutuhkan dan ada pilihan untuk ortu mau menerima atau tidak, krn sayang sel...","Penyalurannya juga hrs lebih selektif , benar"" yang membutuhkan dan ada pilihan untuk ortu mau menerima atau tidak, karena say..."



Penanda regional · indikasi saja


,0
n_train,1


## 06 · Post-processing · fungsi keputusan

Threshold gate memilih probabilitas tertinggi di kelas yang lolos ambang; jika tidak ada yang lolos, kembali ke argmax. Ini berbeda dari sekadar p/threshold, yang ekuivalen dengan bias di log-probability. Bias kelas di sini ditambahkan ke log(p). Parameter dicari untuk macro-F1, bukan accuracy.

In [23]:
from scipy.special import softmax
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

def threshold_predict(prob,thresholds):
    eligible=prob>=thresholds
    pred=np.where(eligible,prob,-1).argmax(1)
    return np.where(eligible.any(axis=1),pred,prob.argmax(1))

def fit_threshold(prob,target):
    thresholds=np.zeros(C)
    for _ in range(2):
        for k in range(C):
            best=macro(target,threshold_predict(prob,thresholds))
            for value in [0,.2,.3,.4,.5,.6,.7,.8]:
                candidate=thresholds.copy();candidate[k]=value
                score=macro(target,threshold_predict(prob,candidate))
                if score>best+1e-10:best,thresholds=score,candidate
    return thresholds

In [24]:
def bias_prob(prob,bias):return softmax(np.log(np.clip(prob,1e-12,1))+bias,axis=1)

def fit_bias(prob,target):
    bias=np.zeros(C)
    for _ in range(2):
        for k in range(C):
            best=macro(target,bias_prob(prob,bias).argmax(1))
            for value in [-1.,-.6,-.3,0,.3,.6,1.]:
                candidate=bias.copy();candidate[k]=value
                score=macro(target,bias_prob(prob,candidate).argmax(1))
                if score>best+1e-10:best,bias=score,candidate
    return bias

In [25]:
def mix(stack,weights):return np.einsum("mnc,m->nc",stack,weights)

def fit_blend(stack,target):
    m=len(stack);rng=np.random.default_rng(SEED)
    candidates=np.vstack([np.eye(m),np.ones((1,m))/m,rng.dirichlet(np.ones(m),ENSEMBLE_CANDIDATES)])
    scores=[macro(target,mix(stack,w).argmax(1)) for w in candidates]
    return candidates[int(np.argmax(scores))]

def temperature_prob(prob,temperature):return softmax(np.log(np.clip(prob,1e-12,1))/temperature,axis=1)
def fit_temperature(prob,target):
    result=minimize_scalar(lambda x:log_loss(target,temperature_prob(prob,np.exp(x)),labels=np.arange(C)),
                           bounds=(np.log(.25),np.log(4.)),method="bounded")
    return float(np.exp(result.x))

In [26]:
def duplicate_prob(prob,keys):
    # Konsistensi berdasarkan exact safe+casefold+spasi, bukan ID pengguna.
    frame=pd.DataFrame(prob,columns=range(C))
    frame["key"]=np.asarray(keys)
    return frame.groupby("key",sort=False)[list(range(C))].transform("mean").to_numpy()

def probability_metrics(target,prob):
    clipped=np.clip(prob,1e-12,1);clipped=clipped/clipped.sum(axis=1,keepdims=True)
    confidence=clipped.max(1);correct=clipped.argmax(1)==target;ece=0.
    bins=np.minimum((confidence*10).astype(int),9)
    for k in range(10):
        mask=bins==k
        if mask.any():ece+=mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    return {"NLL":log_loss(target,clipped,labels=np.arange(C)),
        "Brier_sum":np.square(clipped-np.eye(C)[target]).sum(axis=1).mean(),"ECE_argmax_10bins":ece}

**Sanity check dilakukan di Kaggle sebelum training.** Temperature scalar harus menjaga argmax, setiap transformasi probabilitas harus tetap berjumlah satu. NLL/Brier/ECE mengukur probabilitas; threshold/rule hanya mengubah keputusan label sehingga metrik probabilitasnya tetap merujuk distribusi asal.

In [27]:
fixture=np.full((C*2,C),.2/(C-1));fixture[np.arange(C*2),np.arange(C*2)%C]=.8
fixture_y=np.arange(C*2)%C
assert np.array_equal(temperature_prob(fixture,2.).argmax(1),fixture.argmax(1))
assert np.allclose(bias_prob(fixture,np.zeros(C)),fixture)
assert np.array_equal(threshold_predict(fixture,np.zeros(C)),fixture.argmax(1))
assert np.allclose(mix(np.stack([fixture,fixture]),np.array([.4,.6])),fixture)
assert np.allclose(duplicate_prob(fixture,np.arange(len(fixture))),fixture)
for transformed in [bias_prob(fixture,fit_bias(fixture,fixture_y)),temperature_prob(fixture,fit_temperature(fixture,fixture_y))]:
    assert np.allclose(transformed.sum(axis=1),1)
print("Sanity check post-processing: PASS")

Sanity check post-processing: PASS


## 07 · Rule-based correction yang dibatasi

Aturan template sentimen ini bukan kebenaran universal. Hanya aturan dengan support train >= RULE_MIN_SUPPORT dan agreement pseudo-label train >= RULE_MIN_PRECISION yang diaktifkan. Bukan menggunakan label validation/holdout untuk membuat kamus. Aturan bisa tidak aktif sama sekali; override hanya pada model ber-confidence rendah.

In [28]:
RULES=[("positive",r"^(?:sangat )?(?:bagus|mantap|keren)(?: sekali| banget)?[.! ]*$"),
       ("positive",r"^(?:saya )?(?:setuju|mendukung)(?: program ini| mbg)?[.! ]*$"),
       ("negative",r"^(?:saya )?tidak setuju(?: program ini| mbg)?[.! ]*$"),
       ("negative",r"^(?:program ini |mbg )?(?:buruk|jelek|gagal)(?: sekali| banget)?[.! ]*$")]
rule_text=train.safe.str.casefold();active_rules=[];rule_audit=[]
for label,pattern in RULES:
    hits=rule_text.str.fullmatch(pattern,na=False);support=int(hits.sum())
    agreement=float(train.loc[hits,LABEL_COL].eq(label).mean()) if support else np.nan
    active=support>=RULE_MIN_SUPPORT and agreement>=RULE_MIN_PRECISION and label in classes
    if active:active_rules.append((int(encoder.transform([label])[0]),pattern))
    rule_audit.append({"label":label,"pattern":pattern,"n_train":support,"agreement_train":agreement,"aktif":bool(active)})
show("Audit aturan train",pd.DataFrame(rule_audit))


Audit aturan train


,label,pattern,n_train,agreement_train,aktif
0,positive,^(?:sangat )?(?:bagus|mantap|keren)(?: sekali| banget)?[.! ]*$,6,1.0,False
1,positive,^(?:saya )?(?:setuju|mendukung)(?: program ini| mbg)?[.! ]*$,2,0.5,False
2,negative,^(?:saya )?tidak setuju(?: program ini| mbg)?[.! ]*$,0,NaN,False
3,negative,^(?:program ini |mbg )?(?:buruk|jelek|gagal)(?: sekali| banget)?[.! ]*$,0,NaN,False


In [29]:
def rule_targets(texts):
    result=np.full(len(texts),-1)
    for i,text in enumerate(texts):
        matches={label for label,pattern in active_rules if re.fullmatch(pattern,text.casefold())}
        if len(matches)==1:result[i]=next(iter(matches))
    return result

def rule_predict(prob,targets,gate):
    pred=prob.argmax(1);mask=(targets>=0)&(prob.max(1)<gate)
    return np.where(mask,targets,pred)

def fit_rule_gate(prob,target,rules):
    gates=[0.,.4,.5,.6,.7,.8]
    return max(gates,key=lambda g:macro(target,rule_predict(prob,rules,g)))

## 08 · Trainer bersama

Setiap pasangan model/input dimulai dari pretrained checkpoint yang sama, bukan meneruskan training varian sebelumnya. Bobot loss hanya dari train. Batch XLM-R lebih kecil dengan accumulation agar effective batch sebanding. Model terbaik per keluarga disimpan; optimizer checkpoint sementara dibersihkan untuk menghemat ruang.

In [30]:
import shutil,inspect
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,set_seed
from datasets import Dataset,disable_progress_bars
disable_progress_bars()
from sklearn.utils.class_weight import compute_class_weight
from scipy.special import softmax
weights=compute_class_weight("balanced",classes=np.arange(C),y=train.y.to_numpy())**WEIGHT_POWER
weights=weights/weights.mean()
class WeightedTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        target=inputs.pop("labels");result=model(**inputs)
        loss=torch.nn.functional.cross_entropy(result.logits.float(),target,weight=torch.tensor(weights,dtype=torch.float32,device=result.logits.device))
        return (loss,result) if return_outputs else loss

def encode(texts,tokenizer,labels=None):
    payload={"text":list(texts)}
    if labels is not None:payload["labels"]=list(labels)
    return Dataset.from_dict(payload).map(lambda b:tokenizer(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns="text")

In [31]:
def make_trainer(spec,run,tokenizer,encoded_train,encoded_val):
    model=AutoModelForSequenceClassification.from_pretrained(spec["id"],revision=spec["revision"],trust_remote_code=False,
        num_labels=C,id2label=dict(enumerate(classes)),label2id={label:i for i,label in enumerate(classes)},ignore_mismatched_sizes=True)
    steps=int(np.ceil(len(train)/(spec["batch"]*max(1,torch.cuda.device_count())*spec["accum"]))*EPOCHS)
    args=TrainingArguments(output_dir=str(OUT/"checkpoints"/run),num_train_epochs=EPOCHS,learning_rate=LEARNING_RATE,
        per_device_train_batch_size=spec["batch"],per_device_eval_batch_size=spec["batch"],gradient_accumulation_steps=spec["accum"],
        warmup_steps=int(.1*steps),weight_decay=.01,optim="adamw_torch",fp16=True,
        eval_strategy="epoch",save_strategy="epoch",save_only_model=True,save_total_limit=1,logging_strategy="epoch",
        load_best_model_at_end=True,metric_for_best_model="macro_f1",greater_is_better=True,
        report_to="none",disable_tqdm=True,seed=SEED,data_seed=SEED,dataloader_num_workers=0)
    trainer=WeightedTrainer(model=model,args=args,train_dataset=encoded_train,eval_dataset=encoded_val,
        data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=lambda p:{"macro_f1":macro(p.label_ids,p.predictions.argmax(1))})
    trainer.model_accepts_loss_kwargs=False
    return trainer

In [32]:
def label_metrics(target,pred):
    return {"macro_f1":macro(target,pred),"weighted_f1":f1_score(target,pred,average="weighted",zero_division=0),
            "accuracy":accuracy_score(target,pred),"balanced_acc":balanced_accuracy_score(target,pred)}
records=[];predictions={};best_per_model={};token_audit=[]

def prepare_inputs(spec,view,tokenizer):
    a,b,c=[model_input(frame,spec,view) for frame in [train,val,holdout]]
    ids=tokenizer(a.tolist(),truncation=False)["input_ids"]
    lengths=pd.Series(list(map(len,ids)))
    token_audit.append({"model":spec["name"],"view":view,"token_p50":lengths.median(),"token_p95":lengths.quantile(.95),
        "token_p99":lengths.quantile(.99),"terpotong_%":lengths.gt(MAX_LEN).mean()*100,
        "UNK_%_termasuk_special":100*sum(seq.count(tokenizer.unk_token_id) for seq in ids)/(sum(map(len,ids)) or np.nan)})
    return encode(a,tokenizer,train.y),encode(b,tokenizer,val.y),encode(c,tokenizer)

**Prediksi holdout tanpa label:** logits holdout dihitung saat model masih di GPU agar tidak perlu memuat ulang sepuluh model. Holdout dataset untuk Trainer.predict **tidak membawa labels**; metriknya baru dihitung setelah seluruh model/normalisasi/post-processing dipilih dan disimpan. Tidak ada tuning memakai skor holdout.

In [33]:
def run_experiment(spec,view):
    run=spec["name"]+"__"+view;set_seed(SEED);torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    tokenizer=AutoTokenizer.from_pretrained(spec["id"],revision=spec["revision"],trust_remote_code=False)
    a,b,c=prepare_inputs(spec,view,tokenizer)
    trainer=make_trainer(spec,run,tokenizer,a,b)
    start=time.perf_counter();trainer.train();seconds=time.perf_counter()-start
    vp=softmax(trainer.predict(b).predictions,axis=1)
    predict_start=time.perf_counter();hp=softmax(trainer.predict(c).predictions,axis=1)
    infer_seconds=(time.perf_counter()-predict_start)/len(holdout)*1000
    assert vp.shape==(len(val),C) and hp.shape==(len(holdout),C) and np.isfinite(vp).all() and np.isfinite(hp).all()
    assert np.allclose(vp.sum(axis=1),1,atol=1e-5) and np.allclose(hp.sum(axis=1),1,atol=1e-5)
    predictions[run]={"validation":vp,"holdout":hp}
    row={"model":spec["name"],"model_id":spec["id"],"view":view,"run":run,"fit_seconds":seconds,
        "parameters":trainer.model.num_parameters(),"infer_seconds_per1000":infer_seconds,"gpu_peak_GB":torch.cuda.max_memory_allocated()/1e9,
        **label_metrics(val.y.to_numpy(),vp.argmax(1)),**probability_metrics(val.y.to_numpy(),vp)}
    records.append(row)
    if spec["name"] not in best_per_model or row["macro_f1"]>best_per_model[spec["name"]]["macro_f1"]:
        best_per_model[spec["name"]]=row
        folder=OUT/"models"/spec["name"];trainer.save_model(str(folder));tokenizer.save_pretrained(str(folder))
        (folder/"input_policy.json").write_text(json.dumps({"view":view,"family":spec["family"],"model_id":spec["id"],"revision":spec["revision"]},indent=2))
    np.savez_compressed(OUT/(run+".npz"),validation=vp,holdout=hp)
    pd.DataFrame(trainer.state.log_history).to_csv(OUT/(run+"_training.csv"),index=False)
    (OUT/"progress.json").write_text(json.dumps(records,indent=2,default=float))
    print(json.dumps(row,default=float),flush=True)
    checkpoint=Path(trainer.args.output_dir).resolve()
    del trainer,tokenizer,a,b,c;gc.collect();torch.cuda.empty_cache()
    assert checkpoint.is_relative_to((OUT/"checkpoints").resolve())
    shutil.rmtree(checkpoint)

## 09 · Sepuluh fine-tuning

Urutan: native dan normalized untuk masing-masing lima model. Hyperparameter dasar sama, head klasifikasi baru untuk setiap run. Normalisasi dipilih per keluarga melalui validation macro-F1; tidak memilih lewat holdout. Ini pencarian terbatas, bukan pembuktian model tertentu selalu lebih unggul.

In [34]:
print("Jumlah GPU:",torch.cuda.device_count(),"| eksperimen:",len(MODEL_SPECS)*2)
for spec in MODEL_SPECS:
    for view in ["native","normalized"]:
        print("START",spec["name"],view,flush=True)
        run_experiment(spec,view)
assert len(records)==10 and len(best_per_model)==5
validation_table=pd.DataFrame(records).sort_values("macro_f1",ascending=False)
show("Validation · 10 eksperimen",validation_table.drop(columns=["model_id","parameters"]))
show("Tokenization train per model/input",pd.DataFrame(token_audit))

Jumlah GPU: 2 | eksperimen: 10
START 01_indobertweet native


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.936', 'grad_norm': '11.21', 'learning_rate': '1.492e-05', 'epoch': '1'}


{'eval_loss': '0.7518', 'eval_macro_f1': '0.6594', 'eval_runtime': '6.379', 'eval_samples_per_second': '156.5', 'eval_steps_per_second': '2.508', 'epoch': '1'}


{'loss': '0.6412', 'grad_norm': '4.721', 'learning_rate': '7.538e-06', 'epoch': '2'}


... (104 outputs left out)


Validation · 10 eksperimen


,model,view,run,fit_seconds,infer_seconds_per1000,gpu_peak_GB,macro_f1,weighted_f1,accuracy,balanced_acc,NLL,Brier_sum,ECE_argmax_10bins
2,02_indobert_p1,native,02_indobert_p1__native,201.408,7.597,5.460,0.735,0.823,0.826,0.724,0.470,0.267,0.052
3,02_indobert_p1,normalized,02_indobert_p1__normalized,201.149,7.520,5.460,0.728,0.816,0.819,0.716,0.472,0.270,0.050
1,01_indobertweet,normalized,01_indobertweet__normalized,199.170,7.415,5.295,0.696,0.803,0.805,0.693,0.497,0.279,0.030
0,01_indobertweet,native,01_indobertweet__native,193.189,7.501,5.297,0.686,0.800,0.803,0.681,0.504,0.283,0.028
6,04_indobert_indolem,native,04_indobert_indolem__native,199.524,7.438,5.295,0.660,0.765,0.763,0.668,0.575,0.328,0.050
7,04_indobert_indolem,normalized,04_indobert_indolem__normalized,198.968,7.445,5.295,0.636,0.753,0.751,0.641,0.629,0.360,0.104
4,03_xlm_roberta,native,03_xlm_roberta__native,261.120,9.222,6.717,0.612,0.768,0.788,0.606,0.549,0.306,0.036
5,03_xlm_roberta,normalized,03_xlm_roberta__normalized,260.744,9.186,6.717,0.517,0.736,0.778,0.547,0.561,0.312,0.035
8,05_nusabert,native,05_nusabert__native,199.019,7.370,5.295,0.478,0.688,0.744,0.460,0.705,0.405,0.157
9,05_nusabert,normalized,05_nusabert__normalized,199.115,7.356,5.295,0.421,0.609,0.675,0.441,0.776,0.458,0.118



Tokenization train per model/input


,model,view,token_p50,token_p95,token_p99,terpotong_%,UNK_%_termasuk_special
0,01_indobertweet,native,28.0,135.0,289.64,2.252,2.806
1,01_indobertweet,normalized,28.0,135.0,289.64,2.252,2.806
2,02_indobert_p1,native,26.0,132.1,280.82,2.219,0.596
3,02_indobert_p1,normalized,26.0,132.1,280.82,2.219,0.596
4,03_xlm_roberta,native,30.0,150.0,327.00,2.915,0.007
5,03_xlm_roberta,normalized,30.0,149.0,324.00,2.849,0.007
6,04_indobert_indolem,native,27.0,134.0,287.64,2.219,0.529
7,04_indobert_indolem,normalized,27.0,133.1,287.46,2.219,0.530
8,05_nusabert,native,28.0,143.1,311.64,2.716,0.497
9,05_nusabert,normalized,28.0,143.1,312.46,2.716,0.497


## 10 · Pilih input per model & audit pengaruh normalisasi

Selisih normalized-native dihitung dari validation. Nilai negatif berarti normalisasi merugikan pada run ini. Pilihan input tiap model, model dasar post-processing, dan urutan model dibekukan sebelum holdout metrics.

In [35]:
normalization_comparison=pd.DataFrame(records).pivot(index="model",columns="view",values="macro_f1")
normalization_comparison["delta_normalized_minus_native"]=normalization_comparison.normalized-normalization_comparison.native
show("Apakah normalisasi membantu? · validation",normalization_comparison)
winners=sorted(best_per_model.values(),key=lambda row:row["macro_f1"],reverse=True)
selected_runs=[row["run"] for row in winners]
best_run=selected_runs[0]
stack_val=np.stack([predictions[run]["validation"] for run in selected_runs])
stack_hold=np.stack([predictions[run]["holdout"] for run in selected_runs])
base_val=predictions[best_run]["validation"];base_hold=predictions[best_run]["holdout"]
y_val=val.y.to_numpy()
show("Pemenang per keluarga · validation",pd.DataFrame(winners)[["model","view","macro_f1","NLL","fit_seconds"]])
print("Model dasar post-processing:",best_run)


Apakah normalisasi membantu? · validation


view,native,normalized,delta_normalized_minus_native
model,,,
01_indobertweet,0.686,0.696,0.010
02_indobert_p1,0.735,0.728,-0.007
03_xlm_roberta,0.612,0.517,-0.095
04_indobert_indolem,0.660,0.636,-0.024
05_nusabert,0.478,0.421,-0.057



Pemenang per keluarga · validation


,model,view,macro_f1,NLL,fit_seconds
0,02_indobert_p1,native,0.735,0.470,201.408
1,01_indobertweet,normalized,0.696,0.497,199.170
2,04_indobert_indolem,native,0.660,0.575,199.524
3,03_xlm_roberta,native,0.612,0.549,261.120
4,05_nusabert,native,0.478,0.705,199.019


Model dasar post-processing: 02_indobert_p1__native


## 11 · Cross-fit enam teknik + gabungan

Semua metode memakai fold grup validation yang sama. Threshold/bias/ensemble/rule gate/temperature dipelajari pada bagian tune, dinilai pada bagian check. Duplicate consistency hanya menggabungkan probabilitas exact-text di prediction batch, tidak mengambil label train atau ID pengguna. Temperatur scalar mengubah confidence, bukan argmax. Aturan bisa tetap nonaktif bila dukungan tidak cukup.

In [36]:
val_keys=val.safe.str.casefold().str.replace(r"\s+"," ",regex=True).to_numpy()
hold_keys=holdout.safe.str.casefold().str.replace(r"\s+"," ",regex=True).to_numpy()
val_rules=rule_targets(val.safe.tolist());hold_rules=rule_targets(holdout.safe.tolist())
METHODS=["raw","threshold","class_bias","ensemble","duplicate_consistency","rule_correction","temperature","ensemble_bias"]
oof_label={name:np.empty(len(val),dtype=int) for name in METHODS}
oof_prob={name:np.empty_like(base_val) for name in METHODS}
fold_stats=[]
for tune,check in StratifiedGroupKFold(POST_FOLDS,shuffle=True,random_state=SEED).split(val,y_val,val.dup_group):
    p,target=base_val[tune],y_val[tune];q=base_val[check]
    threshold=fit_threshold(p,target);bias=fit_bias(p,target);blend=fit_blend(stack_val[:,tune],target)
    gate=fit_rule_gate(p,target,val_rules[tune]);temperature=fit_temperature(p,target)
    blended_tune=mix(stack_val[:,tune],blend);blended_check=mix(stack_val[:,check],blend)
    combined_bias=fit_bias(blended_tune,target)
    probs={"raw":q,"threshold":q,"class_bias":bias_prob(q,bias),"ensemble":blended_check,
        "duplicate_consistency":duplicate_prob(q,val_keys[check]),"rule_correction":q,
        "temperature":temperature_prob(q,temperature),"ensemble_bias":bias_prob(blended_check,combined_bias)}
    for name,prob in probs.items():
        pred=threshold_predict(prob,threshold) if name=="threshold" else rule_predict(prob,val_rules[check],gate) if name=="rule_correction" else prob.argmax(1)
        oof_prob[name][check]=prob;oof_label[name][check]=pred
    fold_stats.append({"tune_n":len(tune),"check_n":len(check),"temperature":temperature,"gate":gate,
        "threshold":threshold.tolist(),"bias":bias.tolist(),"ensemble_weights":blend.tolist()})

In [37]:
post_validation=pd.DataFrame([{"method":name,**label_metrics(y_val,oof_label[name]),**probability_metrics(y_val,oof_prob[name])} for name in METHODS])
raw_score=float(post_validation.loc[post_validation.method.eq("raw"),"macro_f1"].iloc[0])
post_validation["delta_macro_f1"]=post_validation.macro_f1-raw_score
show("Post-processing · cross-fit validation",post_validation.sort_values("macro_f1",ascending=False))
print("Duplicate groups validation:",pd.Series(val_keys).value_counts().gt(1).sum(),"| baris rule match:",(val_rules>=0).sum())
assert np.array_equal(oof_label["raw"],oof_label["temperature"])
print("Temperature argmax tetap sama: PASS; perhatikan NLL/Brier/ECE, bukan berharap F1 naik.")
chosen=post_validation.sort_values("macro_f1",ascending=False).iloc[0]
chosen_method=chosen.method if chosen.macro_f1>raw_score+MIN_POST_GAIN else "raw"
print("Metode final dari cross-fit validation:",chosen_method)


Post-processing · cross-fit validation


,method,macro_f1,weighted_f1,accuracy,balanced_acc,NLL,Brier_sum,ECE_argmax_10bins,delta_macro_f1
3,ensemble,0.747,0.832,0.837,0.722,0.451,0.248,0.069,0.013
2,class_bias,0.743,0.823,0.823,0.747,0.491,0.273,0.046,0.008
7,ensemble_bias,0.741,0.819,0.816,0.759,0.504,0.276,0.082,0.007
0,raw,0.735,0.823,0.826,0.724,0.470,0.267,0.052,0.000
5,rule_correction,0.735,0.823,0.826,0.724,0.470,0.267,0.052,0.000
4,duplicate_consistency,0.735,0.823,0.826,0.724,0.470,0.267,0.052,0.000
6,temperature,0.735,0.823,0.826,0.724,0.461,0.262,0.026,0.000
1,threshold,0.734,0.822,0.824,0.729,0.470,0.267,0.052,-0.000


Duplicate groups validation: 13 | baris rule match: 0
Temperature argmax tetap sama: PASS; perhatikan NLL/Brier/ECE, bukan berharap F1 naik.
Metode final dari cross-fit validation: ensemble


## 12 · Bekukan parameter post-processing

Setelah metode dipilih dari cross-fit validation, parameter metode dipasang sekali pada seluruh validation. Tidak ada label holdout dipakai untuk pemilihan bobot, threshold, bias, rule gate, atau temperature.

In [38]:
full_threshold=fit_threshold(base_val,y_val);full_bias=fit_bias(base_val,y_val)
full_blend=fit_blend(stack_val,y_val);full_temperature=fit_temperature(base_val,y_val)
full_gate=fit_rule_gate(base_val,y_val,val_rules)
full_combined_bias=fit_bias(mix(stack_val,full_blend),y_val)
selection={"best_run":best_run,"selected_runs":selected_runs,"method":chosen_method,
    "thresholds":full_threshold.tolist(),"class_bias":full_bias.tolist(),"ensemble_weights":full_blend.tolist(),
    "temperature":full_temperature,"rule_gate":full_gate,"ensemble_bias":full_combined_bias.tolist(),"classes":classes.tolist()}
(OUT/"selection.json").write_text(json.dumps(selection,indent=2,default=float))
print("KEPUTUSAN DIBEKUKAN",json.dumps(selection,indent=2,default=float))

KEPUTUSAN DIBEKUKAN {
  "best_run": "02_indobert_p1__native",
  "selected_runs": [
    "02_indobert_p1__native",
    "01_indobertweet__normalized",
    "04_indobert_indolem__native",
    "03_xlm_roberta__native",
    "05_nusabert__native"
  ],
  "method": "ensemble",
  "thresholds": [
    0.5,
    0.0,
    0.4
  ],
  "class_bias": [
    0.3,
    1.0,
    0.0
  ],
  "ensemble_weights": [
    0.4322783287582675,
    0.11101768364067127,
    0.2003073342633165,
    0.20253296339746985,
    0.05386368994027496
  ],
  "temperature": 1.2007778957139825,
  "rule_gate": 0.0,
  "ensemble_bias": [
    -0.3,
    0.6,
    0.0
  ],
  "classes": [
    "negative",
    "neutral",
    "positive"
  ]
}


## 13 · Holdout · semua model dan post-processing

Sekarang label holdout dipakai hanya untuk laporan. Seluruh tabel mempertahankan pilihan validation; tidak beralih ke metode yang kebetulan terbaik di holdout. NLL/Brier/ECE merujuk probabilitas argmax, sehingga threshold/rule yang hanya mengubah label tidak otomatis memperbaiki probabilitas.

In [39]:
y_hold=holdout.y.to_numpy()
hold_raw=pd.DataFrame([{"model":row["model"],"view":row["view"],"run":row["run"],
    **label_metrics(y_hold,predictions[row["run"]]["holdout"].argmax(1)),
    **probability_metrics(y_hold,predictions[row["run"]]["holdout"])} for row in winners])
show("Lima model · input dipilih validation",hold_raw.drop(columns="run"))
all_hold=pd.DataFrame([{"run":row["run"],**label_metrics(y_hold,predictions[row["run"]]["holdout"].argmax(1))} for row in records])
show("Seluruh 10 run · tidak dipakai memilih input",all_hold)
post_hold_prob={"raw":base_hold,"threshold":base_hold,"class_bias":bias_prob(base_hold,full_bias),
    "ensemble":mix(stack_hold,full_blend),"duplicate_consistency":duplicate_prob(base_hold,hold_keys),
    "rule_correction":base_hold,"temperature":temperature_prob(base_hold,full_temperature),
    "ensemble_bias":bias_prob(mix(stack_hold,full_blend),full_combined_bias)}
post_hold_label={name:threshold_predict(prob,full_threshold) if name=="threshold" else
    rule_predict(prob,hold_rules,full_gate) if name=="rule_correction" else prob.argmax(1) for name,prob in post_hold_prob.items()}


Lima model · input dipilih validation


,model,view,macro_f1,weighted_f1,accuracy,balanced_acc,NLL,Brier_sum,ECE_argmax_10bins
0,02_indobert_p1,native,0.791,0.869,0.868,0.803,0.344,0.196,0.031
1,01_indobertweet,normalized,0.730,0.840,0.840,0.728,0.409,0.227,0.060
2,04_indobert_indolem,native,0.633,0.788,0.781,0.649,0.560,0.317,0.063
3,03_xlm_roberta,native,0.616,0.806,0.821,0.616,0.468,0.255,0.046
4,05_nusabert,native,0.456,0.708,0.759,0.443,0.680,0.390,0.169



Seluruh 10 run · tidak dipakai memilih input


,run,macro_f1,weighted_f1,accuracy,balanced_acc
0,01_indobertweet__native,0.724,0.841,0.841,0.719
1,01_indobertweet__normalized,0.730,0.840,0.840,0.728
2,02_indobert_p1__native,0.791,0.869,0.868,0.803
3,02_indobert_p1__normalized,0.805,0.876,0.875,0.821
4,03_xlm_roberta__native,0.616,0.806,0.821,0.616
5,03_xlm_roberta__normalized,0.554,0.802,0.834,0.579
6,04_indobert_indolem__native,0.633,0.788,0.781,0.649
7,04_indobert_indolem__normalized,0.614,0.766,0.760,0.630
8,05_nusabert__native,0.456,0.708,0.759,0.443
9,05_nusabert__normalized,0.420,0.643,0.700,0.435


In [40]:
post_hold=pd.DataFrame([{"method":name,**label_metrics(y_hold,post_hold_label[name]),**probability_metrics(y_hold,post_hold_prob[name])} for name in METHODS])
raw_hold_score=macro(y_hold,post_hold_label["raw"])
post_hold["delta_macro_f1_vs_raw"]=post_hold.macro_f1-raw_hold_score
show("Eksperimen post-processing · holdout",post_hold)
print("Rule match holdout:",(hold_rules>=0).sum(),"| label berubah karena rule:",(post_hold_label["rule_correction"]!=post_hold_label["raw"]).sum())
print("Duplikat exact-key holdout:",pd.Series(hold_keys).value_counts().gt(1).sum(),"grup")
final_pred=post_hold_label[chosen_method];final_prob=post_hold_prob[chosen_method]
show("Pilihan final sesuai validation",post_hold.loc[post_hold.method.eq(chosen_method)])
assert np.array_equal(post_hold_label["raw"],post_hold_label["temperature"])


Eksperimen post-processing · holdout


,method,macro_f1,weighted_f1,accuracy,balanced_acc,NLL,Brier_sum,ECE_argmax_10bins,delta_macro_f1_vs_raw
0,raw,0.791,0.869,0.868,0.803,0.344,0.196,0.031,0.000
1,threshold,0.789,0.868,0.866,0.811,0.344,0.196,0.031,-0.002
2,class_bias,0.773,0.856,0.851,0.826,0.383,0.212,0.021,-0.018
3,ensemble,0.778,0.866,0.868,0.764,0.381,0.201,0.101,-0.013
4,duplicate_consistency,0.791,0.869,0.868,0.803,0.344,0.196,0.031,0.000
5,rule_correction,0.791,0.869,0.868,0.803,0.344,0.196,0.031,0.000
6,temperature,0.791,0.869,0.868,0.803,0.357,0.198,0.046,0.000
7,ensemble_bias,0.758,0.851,0.840,0.835,0.458,0.244,0.114,-0.033


Rule match holdout: 0 | label berubah karena rule: 0
Duplikat exact-key holdout: 4 grup

Pilihan final sesuai validation


,method,macro_f1,weighted_f1,accuracy,balanced_acc,NLL,Brier_sum,ECE_argmax_10bins,delta_macro_f1_vs_raw
3,ensemble,0.778,0.866,0.868,0.764,0.381,0.201,0.101,-0.013


## 14 · Precision, recall, F1 & confusion

Tabel seluruh model/raw dan metode post-processing menunjukkan kelas yang dikorbankan atau terbantu. Confusion: baris aktual, kolom prediksi. Tidak ada grafik otomatis.

In [41]:
per_class=[]
reported={row["model"]:predictions[row["run"]]["holdout"].argmax(1) for row in winners}
reported.update({"post_"+name:pred for name,pred in post_hold_label.items()})
for name,pred in reported.items():
    result=classification_report(y_hold,pred,labels=np.arange(C),target_names=classes,output_dict=True,zero_division=0)
    per_class.extend({"model/metode":name,"kelas":label,**result[label]} for label in classes)
per_class=pd.DataFrame(per_class)
show("Per-class metrics",per_class)
cm=pd.DataFrame(confusion_matrix(y_hold,final_pred,labels=np.arange(C)),index=classes,columns=classes)
show("Confusion final jumlah",cm)
show("Confusion final % per kelas aktual",cm.div(cm.sum(axis=1),axis=0)*100)


Per-class metrics


,model/metode,kelas,precision,recall,f1-score,support
0,02_indobert_p1,negative,0.917,0.914,0.916,690.0
1,02_indobert_p1,neutral,0.624,0.726,0.671,73.0
2,02_indobert_p1,positive,0.805,0.768,0.786,220.0
3,01_indobertweet,negative,0.913,0.894,0.903,690.0
4,01_indobertweet,neutral,0.587,0.507,0.544,73.0
5,01_indobertweet,positive,0.705,0.782,0.741,220.0
6,04_indobert_indolem,negative,0.889,0.855,0.871,690.0
7,04_indobert_indolem,neutral,0.310,0.425,0.358,73.0
8,04_indobert_indolem,positive,0.671,0.668,0.670,220.0
9,03_xlm_roberta,negative,0.884,0.894,0.889,690.0



Confusion final jumlah


,negative,neutral,positive
negative,639,13,38
neutral,19,43,11
positive,43,6,171



Confusion final % per kelas aktual


,negative,neutral,positive
negative,92.609,1.884,5.507
neutral,26.027,58.904,15.068
positive,19.545,2.727,77.727


## 15 · Analisis kelemahan & indikasi multilingual

Penanda regional/campur kode adalah indikasi, bukan label bahasa terverifikasi. Support kecil tidak cukup membuktikan NusaBERT/XLM-R unggul untuk bahasa tertentu. F1 slice hanya memakai kelas aktual yang hadir; selalu lihat n/error rate.

In [42]:
analysis=holdout.copy();analysis["prediksi"]=encoder.inverse_transform(final_pred);analysis["salah"]=final_pred!=y_hold
analysis["prob_label_dipilih"]=final_prob[np.arange(len(final_pred)),final_pred]
analysis["panjang"]=pd.cut(analysis.safe.str.split().str.len(),[-1,5,15,40,100,np.inf])
for name in ["negasi","slang_contoh","url","emoji/simbol"]:
    analysis[name]=analysis[TEXT_COL].astype(str).str.count(NOISE[name],flags=re.I).gt(0)
sets=analysis.safe.str.casefold().str.findall(r"[^\W\d_]+").map(set)
analysis["regional_indikator"]=sets.map(lambda s:len(s&REGIONAL_MARKERS)>=2)
en=set("the this that with would should have because very really".split());ind=set("yang tidak bukan belum dengan untuk karena sudah".split())
analysis["campur_indikator"]=sets.map(lambda s:len(s&en)>=2 and len(s&ind)>=2)
if CONF_COL in analysis:analysis["confidence_label"]=pd.cut(pd.to_numeric(analysis[CONF_COL],errors="coerce"),[-np.inf,.6,.8,.95,np.inf])
slice_columns=["panjang","negasi","slang_contoh","url","emoji/simbol","regional_indikator","campur_indikator"]
if CONF_COL in analysis:slice_columns.append("confidence_label")

In [43]:
slice_rows=[]
for name,pred in {**{row["model"]:predictions[row["run"]]["holdout"].argmax(1) for row in winners},"final_selected":final_pred}.items():
    for column in slice_columns:
        for value,ids in analysis.groupby(column,observed=True,dropna=False).indices.items():
            target=y_hold[ids];p=pred[ids]
            slice_rows.append({"model":name,"slice":column,"nilai":str(value),"n":len(ids),"error_%":100*(p!=target).mean(),
                "macro_f1_kelas_hadir":f1_score(target,p,labels=np.unique(target),average="macro",zero_division=0)})
slices=pd.DataFrame(slice_rows)
show("Kelemahan pilihan final",slices[slices.model.eq("final_selected")])
show("Model per indikasi bahasa",slices[slices["slice"].isin(["regional_indikator","campur_indikator"])])
cols=[TEXT_COL,LABEL_COL,"prediksi","prob_label_dipilih"]+([CONF_COL] if CONF_COL in analysis else [])
show("Salah dengan probabilitas label pilihan tertinggi · bukan bukti label sumber benar",analysis[analysis.salah].sort_values("prob_label_dipilih",ascending=False)[cols].head(TOP))


Kelemahan pilihan final


,model,slice,nilai,n,error_%,macro_f1_kelas_hadir
95,final_selected,panjang,"(-1.0, 5.0]",119,16.807,0.811
96,final_selected,panjang,"(5.0, 15.0]",329,14.894,0.782
97,final_selected,panjang,"(15.0, 40.0]",334,12.575,0.653
98,final_selected,panjang,"(40.0, 100.0]",156,8.974,0.864
99,final_selected,panjang,"(100.0, inf]",45,11.111,0.691
100,final_selected,negasi,False,599,16.027,0.783
101,final_selected,negasi,True,384,8.854,0.636
102,final_selected,slang_contoh,False,591,14.890,0.776
103,final_selected,slang_contoh,True,392,10.714,0.794
104,final_selected,url,False,968,13.017,0.781



Model per indikasi bahasa


,model,slice,nilai,n,error_%,macro_f1_kelas_hadir
13,02_indobert_p1,regional_indikator,False,983,13.225,0.791
14,02_indobert_p1,campur_indikator,False,983,13.225,0.791
32,01_indobertweet,regional_indikator,False,983,15.972,0.730
33,01_indobertweet,campur_indikator,False,983,15.972,0.730
51,04_indobert_indolem,regional_indikator,False,983,21.872,0.633
52,04_indobert_indolem,campur_indikator,False,983,21.872,0.633
70,03_xlm_roberta,regional_indikator,False,983,17.904,0.616
71,03_xlm_roberta,campur_indikator,False,983,17.904,0.616
89,05_nusabert,regional_indikator,False,983,24.110,0.456
90,05_nusabert,campur_indikator,False,983,24.110,0.456



Salah dengan probabilitas label pilihan tertinggi · bukan bukti label sumber benar


,Comment,Sentiment,prediksi,prob_label_dipilih,Confidence Score
4030,"Yang kontrak terhadap MBG lebih banyak sebenarnya,, dan kami tidak bisa di pengaruhi dengan kata kata",neutral,negative,0.875,0.520
2687,"Satgas MBG di daerah tidak bekerja, untuk mengawasi kualitas makanan. Tentu yg sampai ke presiden yg bagus bagus aja.",positive,negative,0.863,0.582
2480,"<b>PROGRAM MBG HARUS DIBUBARKAN DAN DI HENTIKAN.1 HARI 1,3 T,1 TH 364 T LEBIH.DANA SEBESAR ITU LEBIH BAIK UNTUK PENDIDIKAN GRA...",positive,negative,0.827,0.944
4089,Saya orang biasa saya suka mbg karna Saya tidak punya uang untuk membekali anak sekola.. Saya tidak pernah mendapatkan bantuan...,positive,negative,0.810,0.878
2839,REALISTIS bung... trimakasih kuliahnya.,negative,positive,0.790,0.532
1737,Sukses sll prof 💪🤝,negative,positive,0.783,0.569
3832,"MBG...yng tepat seperti yng Ahok dan kdm bilang, kasih uangnya ke ortu, JD ga ada peluang dikorupsi",positive,negative,0.782,0.850
3000,"Saya rakyat biasa dari awal ide MBG yg di buat si Wowo kami udah tau permainan apa yg sedang di buatnya,dia takkan peduli nega...",positive,negative,0.777,0.836
2657,Gratis kok pake pajak rakyat.dimana gratisnya,positive,negative,0.777,0.502
4243,Kemaren ada artis kaya dukung demo...,positive,negative,0.777,0.441


In [44]:
raw_preds=np.stack([predictions[run]["holdout"].argmax(1) for run in selected_runs])
analysis["jumlah_label_berbeda"]=np.apply_along_axis(lambda p:len(set(p)),0,raw_preds)
show("Disagreement lima model",distribution(analysis.jumlah_label_berbeda))
show("Contoh tidak sepakat",analysis[analysis.jumlah_label_berbeda.ge(3)][cols+["jumlah_label_berbeda"]].head(TOP))
worst=per_class[per_class["model/metode"].eq("post_"+chosen_method)].sort_values("f1-score").iloc[0]
print("Kelas terlemah:",worst.kelas,"| F1:",round(worst["f1-score"],3),"| support:",int(worst.support))
print("Perubahan post-processing pada holdout:",round(macro(y_hold,final_pred)-raw_hold_score,4),"macro-F1; pilihan tidak diganti berdasarkan hasil ini.")


Disagreement lima model


,n,%
jumlah_label_berbeda,,
1,591,60.122
2,334,33.978
3,58,5.900



Contoh tidak sepakat


,Comment,Sentiment,prediksi,prob_label_dipilih,Confidence Score,jumlah_label_berbeda
146,Pemaparan yg lugas &amp; objektif,positive,positive,0.725,0.850,3
186,"Saya punya peternakan lele skala rumah tangga dg panen per siklus 1 ton, tapi sulit masuk SPPG",negative,neutral,0.394,0.484,3
277,"Saya ingin MBG di uang kan, supaya saya punya uang u tuk membeli beras",neutral,positive,0.415,0.725,3
433,"Menambahkan pak, sy dengar banyak oknum sppg mintanya nota kosong ketika belanja bahan makanan pak",negative,negative,0.534,0.567,3
464,🤝🤝🤝🤝,negative,positive,0.440,0.960,3
500,Dapur mbg nego sadis ke petani ke peternak ayam . . Bapak saya pedagang daging ayam dapur MBG minta harga 35.000 perkilo dagin...,negative,neutral,0.358,0.656,3
904,bang gema and bang bennix ❤❤❤❤,neutral,positive,0.465,0.817,3
1052,Mudah mudahan di dengar pemerintah dan dijalankan,positive,positive,0.540,0.876,3
1056,Bapak ini bisa diusulkan masuk sebagai orang BGN. Bisa sebagai analis....,neutral,neutral,0.481,0.902,3
1057,Ini baru namanya profesor 👍,negative,negative,0.478,0.581,3


Kelas terlemah: neutral | F1: 0.637 | support: 73
Perubahan post-processing pada holdout: -0.0132 macro-F1; pilihan tidak diganti berdasarkan hasil ini.


## 16 · Ekspor bukti run & penggunaan ulang

Model terbaik/input policy tiap keluarga disimpan di hasil_v2/models. Lima model itu masih hanya dilatih pada train. Untuk inference baru, terapkan safe cleaning + input_policy.json yang sama; ensemble mengikuti selected_runs/weights dan urutan classes pada selection.json. Duplicate consistency bergantung pada batch inference. Untuk produksi, refit setelah eksperimen disepakati.

In [45]:
validation_table.to_csv(OUT/"validation_runs.csv",index=False)
normalization_comparison.to_csv(OUT/"normalization_comparison.csv")
pd.DataFrame(token_audit).to_csv(OUT/"tokenization.csv",index=False)
post_validation.to_csv(OUT/"post_validation.csv",index=False);post_hold.to_csv(OUT/"post_holdout.csv",index=False)
hold_raw.to_csv(OUT/"holdout_models.csv",index=False);all_hold.to_csv(OUT/"holdout_all_runs.csv",index=False)
per_class.to_csv(OUT/"per_class.csv",index=False);slices.to_csv(OUT/"error_slices.csv",index=False)
pd.DataFrame(rule_audit).to_csv(OUT/"rule_audit.csv",index=False)
analysis[["row_id",TEXT_COL,LABEL_COL,"prediksi","salah","prob_label_dipilih"]].to_csv(OUT/"holdout_predictions.csv",index=False)

In [46]:
import importlib.metadata as metadata
manifest={"experiment":"multiclass_pretrained_v2","seed":SEED,"raw_rows":len(raw),"rows":len(data),
    "split_rows":{"train":len(train),"validation":len(val),"holdout":len(holdout)},"split_sha256":hashlib.sha256(split.sort_values("row_id").to_csv(index=False).encode()).hexdigest(),
    "dataset_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),"model_specs":MODEL_SPECS,"runs_completed":len(records),
    "epochs":EPOCHS,"max_len":MAX_LEN,"learning_rate":LEARNING_RATE,"weight_power":WEIGHT_POWER,
    "selection":selection,"final_holdout":label_metrics(y_hold,final_pred),"final_probability_metrics":probability_metrics(y_hold,final_prob),
    "gpu":torch.cuda.get_device_name(0),"gpu_count":torch.cuda.device_count(),"elapsed_seconds":time.perf_counter()-RUN_START,
    "versions":{p:metadata.version(p) for p in ["torch","transformers","pandas","scikit-learn","tokenizers"]}}
assert len(records)==10 and len(winners)==5 and all(row["gpu_peak_GB"]>0 for row in records)
(OUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2,default=float))
print(json.dumps(manifest,indent=2,default=float))

{
  "experiment": "multiclass_pretrained_v2",
  "seed": 42,
  "raw_rows": 5035,
  "rows": 5000,
  "split_rows": {
    "train": 3019,
    "validation": 998,
    "holdout": 983
  },
  "split_sha256": "6bd611a55f0907b9917ba0e4381c491044210762a926cd62ddba75a29ec34456",
  "dataset_sha256": "a0283b93e120dd5c411bd37117c38879da21cdb39c6c7ec4c9fafd11d7956fba",
  "model_specs": [
    {
      "name": "01_indobertweet",
      "id": "indolem/indobertweet-base-uncased",
      "family": "tweet",
      "batch": 32,
      "accum": 1,
      "revision": "32e28c05b47e33b6675d2670a1745c50a65e987a"
    },
    {
      "name": "02_indobert_p1",
      "id": "indobenchmark/indobert-base-p1",
      "family": "uncased",
      "batch": 32,
      "accum": 1,
      "revision": "c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2"
    },
    {
      "name": "03_xlm_roberta",
      "id": "FacebookAI/xlm-roberta-base",
      "family": "cased",
      "batch": 16,
      "accum": 2,
      "revision": "e73636d4f797dec63c3081bb6ed5c7b

## Sumber & interpretasi
- [IndoBERTweet](https://huggingface.co/indolem/indobertweet-base-uncased), [IndoBERT p1](https://huggingface.co/indobenchmark/indobert-base-p1), [XLM-R](https://huggingface.co/FacebookAI/xlm-roberta-base), [IndoBERT IndoLEM](https://huggingface.co/indolem/indobert-base-uncased), [NusaBERT](https://huggingface.co/LazarusNLP/NusaBERT-base). Revision dipin di konfigurasi.
- [Model card IndoBERTweet](https://huggingface.co/indolem/indobertweet-base-uncased) menetapkan lowercase, @USER/HTTPURL, dan emoji ke teks. Perlakuan konservatif pada model lain adalah pilihan eksperimen; bukan klaim normalisasi wajib dari model card.
- [Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599): temperature scaling. Temperature positif tunggal menjaga argmax multiclass; manfaatnya dievaluasi lewat NLL/Brier/ECE.
- [scikit-learn leakage](https://scikit-learn.org/stable/common_pitfalls.html): seluruh keputusan/tuning dipisahkan dari laporan holdout. Cross-fit post-processing bukan jaminan validation bebas selection bias.
- Brier_sum = rata-rata jumlah squared error seluruh kelas; ECE argmax memakai 10 bin confidence. Rule/threshold yang tidak mengubah distribusi tidak mengubah NLL/Brier/ECE_argmax.
- Skor benchmark pseudo-label dan subset indikasi bahasa tidak membuktikan akurasi pada label manusia atau semua bahasa daerah. Repeated group CV/multiple seed dan test eksternal beranotasi manusia diperlukan untuk klaim generalisasi lebih kuat.

In [47]:
import nbformat
from IPython import get_ipython
executed=Path("/kaggle/working/__notebook__.ipynb")
if executed.exists():
    finished=nbformat.read(executed,as_version=4)
    finished.metadata.pop("papermill",None)
    for cell in finished.cells:cell.metadata.pop("papermill",None)
    finished.cells[-1].execution_count=get_ipython().execution_count
    nbformat.write(finished,"/kaggle/working/10_multi_kelas_v2_pretrained.ipynb")